In [ ]:
import time
import h5py
import matplotlib.pyplot as plt
import numpy as np
from OOPAO.calibration.InteractionMatrix import InteractionMatrix

try:
    import tomllib
except ImportError:  # Python < 3.11
    import tomli as tomllib

In [ ]:
with open("..\instruments\example_instrument.toml", "rb") as f:
    config = tomllib.load(f)

In [ ]:
# %%
plt.ion()
# number of subaperture for the WFS
n_subaperture = config['wfs']['n_lenslets_across']


# %%-----------------------     TELESCOPE   ----------------------------------
from OOPAO.Telescope import Telescope

# create the Telescope object
tel = Telescope(resolution           = 6*n_subaperture,                          # resolution of the telescope in [pix]
                diameter             = config['telescope']['diameter_m'],                                        # diameter in [m]        
                samplingTime         = 1/1000,                                   # Sampling time in [s] of the AO loop
                centralObstruction   = config['telescope']['obstruction_ratio'],                                      # Central obstruction in [%] of a diameter 
                display_optical_path = False,                                    # Flag to display optical path
                fov                  = 0)                                     # field of view in [arcsec]. If set to 0 (default) this speeds up the computation of the phase screens but is uncompatible with off-axis targets


#%% -----------------------     NGS   ----------------------------------
from OOPAO.Source import Source

# create the Natural Guide Star object
ngs = Source(optBand     = 'R',           # Optical band (see photometry.py)
             magnitude   = 5)

# combine the NGS to the telescope using '*'
ngs*tel

# create the Scientific Target object located at 10 arcsec from the  ngs
src = Source(optBand     = 'J2',           # Optical band (see photometry.py)
             magnitude   = 5)

# combine the SRC to the telescope using '*'
src*tel


#%% -----------------------     ATMOSPHERE   ----------------------------------
from OOPAO.Atmosphere import Atmosphere
           
# create the Atmosphere object
atm = Atmosphere(telescope     = tel,                               # Telescope                              
                 r0            = 0.05,                              # Fried Parameter [m]
                 L0            = 25,                                # Outer Scale [m]
                 fractionalR0  = [0.45 ,0.1  ,0.1  ,0.25  ,0.1   ], # Cn2 Profile
                 windSpeed     = [10   ,12   ,11   ,15    ,20    ], # Wind Speed in [m]
                 windDirection = [0    ,72   ,144  ,216   ,288   ], # Wind Direction in [degrees]
                 altitude      = [0    ,1000 ,5000 ,10000 ,12000 ]) # Altitude Layers in [m]


# initialize atmosphere with current Telescope
atm.initializeAtmosphere(tel)

# The phase screen can be updated using atm.update method (Temporal sampling given by tel.samplingTime)
atm.update()

#%% -----------------------     DEFORMABLE MIRROR   ----------------------------------
from OOPAO.DeformableMirror import DeformableMirror
from OOPAO.MisRegistration import MisRegistration

# mis-registrations object (rotation, shifts..)
misReg = MisRegistration()
misReg.shiftX = 0           # in [m]
misReg.shiftY = 0           # in [m]
misReg.rotationAngle = 0    # in [deg]


# specifying a given number of actuators along the diameter: 
nAct = config['dm']['actuators_in_diameter']
    
dm = DeformableMirror(telescope  = tel,                        # Telescope
                    nSubap       = nAct-1,                     # number of subaperture of the system considered (by default the DM has n_subaperture + 1 actuators to be in a Fried Geometry)
                    mechCoupling = 0.35,                       # Mechanical Coupling for the influence functions
                    misReg       = misReg,                     # Mis-registration associated 
                    coordinates  = None,                       # coordinates in [m]. Should be input as an array of size [n_actuators, 2] 
                    sign         = 1e-5,                       # Stroke
                    pitch        = tel.D/nAct)                 # inter actuator distance. Only used to compute the influence function coupling. The default is based on the n_subaperture value. 


#%% -----------------------     Pyramid WFS   ----------------------------------
from OOPAO.Pyramid import Pyramid

# make sure that the ngs is propagated to the wfs
ngs*tel

wfs = Pyramid(nSubap            = n_subaperture,                # number of subaperture = number of pixel accros the pupil diameter
              telescope         = tel,                          # telescope object
              lightRatio        = 0.5,                          # flux threshold to select valid sub-subaperture
              modulation        = config['wfs']['modulation'],                            # Tip tilt modulation radius
              binning           = 1,                            # binning factor (applied only on the )
              n_pix_separation  = 2,                            # number of pixel separating the different pupils
              n_pix_edge        = 1,                            # number of pixel on the edges of the pupils
              postProcessing    = 'fullFrame_incidence_flux')  # slopesMap_incidence_flux, fullFrame_incidence_flux (see documentation)



#%% Adjust the flux considering number of photons per subap.

n_photons_per_subap = 1000

surface_telescope = tel.pixelArea* tel.pixelSize*tel.pixelSize

if wfs.postProcessing[:10] == 'slopesMaps':    
    n_valid_subap = np.sum(wfs.validSignal)/2
else:
    n_valid_subap = np.sum(wfs.validSignal)/4

ngs.nPhoton = n_photons_per_subap / tel.samplingTime / (surface_telescope/n_valid_subap)       # nPhoton = # photons per s per m2

ngs*tel*wfs



#%% -----------------------     Modal Basis - Zernike  ----------------------------------
from OOPAO.Zernike import Zernike

#% ZERNIKE Polynomials
# create Zernike Object
Z = Zernike(tel,50)
# compute polynomials for given telescope
Z.computeZernike(tel)

# # mode to command matrix to project Zernike Polynomials on DM
M2C_zernike = np.linalg.pinv(np.squeeze(dm.modes[tel.pupilLogical,:]) * 2 * np.pi / ngs.wavelength)@Z.modes
C2Z = np.linalg.pinv(M2C_zernike)

#%% -----------------------     Modal Basis - KL Basis  ----------------------------------


from OOPAO.calibration.compute_KL_modal_basis import compute_KL_basis
# use the default definition of the KL modes with forced Tip and Tilt. For more complex KL modes, consider the use of the compute_KL_basis function. 
M2C_KL = compute_KL_basis(tel,
                          atm,
                          dm,
                          lim = 0) # inversion stability criterion

#%% -----------------------     Calibration: Interaction Matrix  ----------------------------------

# amplitude of the modes in m
stroke=1e-9
# zonal Interaction Matrix
M2C_zonal = np.eye(dm.nValidAct)

# modal Interaction Matrix for 300 modes
M2C_modal = M2C_KL[:,:300]

# swap to geometric WFS for the calibration
ngs**tel*wfs # make sure that the proper source is propagated to the WFS
# zonal interaction matrix
calib_modal = InteractionMatrix(ngs            = ngs,
                                atm            = atm,
                                tel            = tel,
                                dm             = dm,
                                wfs            = wfs,   
                                M2C            = M2C_modal, # M2C matrix used 
                                stroke         = stroke,    # stroke for the push/pull in M2C units
                                nMeasurements  = 12,        # number of simultaneous measurements
                                noise          = 'off',     # disable wfs.cam noise 
                                display        = True,      # display the time using tqdm
                                single_pass    = True)      # only push to compute the interaction matrix instead of push-pull



#%% Define instrument and WFS path detectors
from OOPAO.Detector import Detector
# instrument path
src_cam = Detector(tel.resolution,
                    readoutNoise    = config['science_camera']['ron'],  # readout of the detector in [e-/pixel]
                    QE              = 1,                   # quantum efficiency
                    psf_sampling    = config['science_camera']['sampling_at_calibration'])
src_cam.integrationTime = tel.samplingTime # exposure time for the PSF


# WFS path
ngs_cam = Detector(tel.resolution*2)
ngs_cam.psf_sampling = 4
ngs_cam.integrationTime = tel.samplingTime

ngs**tel*ngs_cam
ngs_psf_ref = ngs_cam.frame.copy()

src**tel*src_cam

src_psf_ref = src_cam.frame.copy()

#%%  Closed loop simulation
from OOPAO.tools.tools import strehlMeter

In [ ]:
dm.coefs = M2C_KL[:,0]
plt.imshow(dm.OPD)
plt.colorbar()

In [ ]:
wfs_frames = []
dm_commands = []
psf_frames = []
wfs_measurements = []
loop_status_list = []

wfs_timestamps = []
dm_timestamps = []
psf_timestamps = []

# These are the calibration data used to close the loop
calib_CL = calib_modal
M2C_CL = M2C_modal
reconstructor = M2C_CL@calib_CL.M 

# initialize Telescope DM commands
dm.coefs=0
loop_status = 0

# You can update the the atmosphere parameter on the fly
atm.r0 = 0.05
atm.windSpeed = list(np.random.randint(5,20,atm.nLayer))
atm.windDirection = list(np.random.randint(0,360,atm.nLayer))

# To make sure to always replay the same turbulence, generate a new phase screen for the atmosphere and combine it with the Telescope
atm.generateNewPhaseScreen(seed=12)

# combine telescope with atmosphere
tel+atm

# propagate both sources
ngs**atm*tel*ngs_cam
src**atm*tel*src_cam

# loop parameters
warmup = 20
nLoop = 1000  # number of iterations
gainCL = 0.4  # integrator gain
leakCL = 1.0 # integrator leak
wfs.cam.photonNoise = False  # enable photon noise on the WFS camera
display = True  # enable the display
frame_delay = 2  # number of frame delay

# variables used to to save closed-loop data data
SR_ngs = np.zeros(nLoop+warmup)
SR_src = np.zeros(nLoop+warmup)

wfe_atmosphere = np.zeros(nLoop+warmup)
wfe_residual_SRC = np.zeros(nLoop+warmup)
wfe_residual_NGS = np.zeros(nLoop+warmup)
wfsSignal = np.arange(0, wfs.nSignal)*0  # buffer to simulate the loop delay

for i in range(nLoop + warmup):
    a = time.time()
    # update phase screens => overwrite tel.OPD and consequently tel.src.phase
    atm.update()
    # save the wave-front error of the incoming turbulence within the pupil
    wfe_atmosphere[i] = np.std(tel.OPD[np.where(tel.pupil > 0)])*1e9
    # propagate light from the ngs through the atmosphere, telescope, DM to the WFS and ngs camera
    ngs**atm*tel*dm*wfs*ngs_cam
    # propagate to the focal plane camera
    wfs*wfs.focal_plane_camera
    # save residuals corresponding to the ngs
    wfe_residual_NGS[i] = np.std(tel.OPD[np.where(tel.pupil > 0)])*1e9
    # save Strehl ratio from the PSF image
    SR_ngs[i] = strehlMeter(PSF=ngs_cam.frame, tel=tel, PSF_ref=ngs_psf_ref, display=False)
    # save the OPD seen by the ngs
    OPD_NGS = ngs.OPD.copy()
    if display:
        NGS_PSF = np.log10(np.abs(ngs_cam.frame))

    # propagate light from the src through the atmosphere, telescope, DM to the src camera
    src**atm*tel*dm*src_cam
    # save residuals corresponding to the SRC
    wfe_residual_SRC[i] = np.std(tel.OPD[np.where(tel.pupil > 0)])*1e9
    # save the OPD seen by the src
    OPD_SRC = src.OPD.copy()
    # save Strehl ratio from the PSF image
    SR_src[i] = strehlMeter(PSF=src_cam.frame, tel=tel, PSF_ref=src_psf_ref, display=False)

    # store the slopes after propagating to the WFS <=> 1 frames delay
    if frame_delay == 1:
        wfsSignal = wfs.signal

    # apply the commands on the DM
    dmResidual = np.matmul(reconstructor, wfsSignal)
    if loop_status:
        dm.coefs = leakCL * dm.coefs - gainCL*dmResidual

    # store the slopes after computing the commands <=> 2 frames delay
    if frame_delay == 2:
        wfsSignal = wfs.signal
    # print('Elapsed time: ' + str(time.time()-a) + ' s')

    if i == 10:
        loop_status = 1

    print('-----------------------------------')
    print('Loop'+str(i) + '/' + str(nLoop+warmup))
    print('NGS: Strehl ratio [%] : ', np.round(SR_ngs[i],1), ' WFE [nm] : ', np.round(wfe_residual_NGS[i],2))
    print('SRC: Strehl ratio [%] : ', np.round(SR_src[i],1), ' WFE [nm] : ', np.round(wfe_residual_SRC[i],2))

    if i > warmup:
        wfs_frames.append(wfs.cam.frame)
        dm_commands.append(dm.coefs)
        psf_frames.append(src_cam.frame)
        wfs_measurements.append(dmResidual)
        loop_status_list.append(loop_status)
        wfs_timestamps.append(time.time())
        dm_timestamps.append(time.time())
        psf_timestamps.append(time.time())
    
#%% Closed Loop data analysis

plt.figure()
plt.plot(np.arange(nLoop+warmup)*tel.samplingTime, wfe_atmosphere, label='Turbulence')
plt.plot(np.arange(nLoop+warmup)*tel.samplingTime, wfe_residual_NGS, label='NGS')
plt.plot(np.arange(nLoop+warmup)*tel.samplingTime, wfe_residual_SRC, label='SRC')
plt.legend()
plt.xlabel('Time [s]')
plt.ylabel('WFE [nm]')

plt.figure()
plt.plot(np.arange(nLoop+warmup)*tel.samplingTime, SR_ngs, label='NGS@' + str(np.round(1e9*ngs.wavelength,0)) + ' nm')
plt.plot(np.arange(nLoop+warmup)*tel.samplingTime, SR_src, label='SRC@' + str(np.round(1e9*src.wavelength,0)) + ' nm')
plt.legend()
plt.xlabel('Time [s]')
plt.ylabel('SR [%]')

psf_frames = np.array(psf_frames)
dm_commands = np.array(dm_commands)
wfs_frames = np.array(wfs_frames[::50])
wfs_measurements = np.array(wfs_measurements)
loop_status_list = np.array(loop_status_list)
wfs_timestamps = np.array(wfs_timestamps)
dm_timestamps = np.array(dm_timestamps)
psf_timestamps = np.array(psf_timestamps)


In [ ]:
plt.plot(dm_commands[0])

In [ ]:
wfs_frames = wfs_frames
psf_frames=psf_frames
dm_commands=dm_commands
m2c=M2C_CL
WFS_PUP=wfs.valid_signal_2D
loop_gain=gainCL
loop_leak=leakCL
wfs_fps=int(1/tel.samplingTime)
wfs_gain=1
sci_dit=src_cam.integrationTime
sci_fps=wfs_fps
sci_gain=wfs_fps

In [ ]:
from astroquery.simbad import Simbad
from astropy.coordinates import SkyCoord, EarthLocation, AltAz
from astropy.time import Time
import astropy.units as u

Simbad.add_votable_fields('V', 'R', 'J', 'H','ra', 'dec')
location = EarthLocation(lat=43.9308333*u.deg, lon=5.71333*u.deg, height=650*u.m)

target = "hip87585"

result = Simbad.query_object(target)

if result:
    print(result)
    ra_str = result['ra'][0]     # e.g., '18 36 56.336'
    dec_str = result['dec'][0]   # e.g., '+38 47 01.28'
    Vmag = result['V'][0]
    Rmag = result['R'][0]
    Hmag = result['H'][0]
    Jmag = result['J'][0] 
    coord = SkyCoord(ra=ra_str, dec=dec_str, unit=(u.deg, u.deg), frame='icrs')
    obstime = Time.now()
    altaz_frame = AltAz(obstime=obstime, location=location)
    altaz = coord.transform_to(altaz_frame)
    alt = altaz.alt
    print(f"Altitude (elevation): {altaz.alt:.2f}")
    print(f"Azimuth: {altaz.az:.2f}")
else:
    Vmag=0
    Rmag=0
    Hmag=0
    Jmag=0
    alt = 0
    print("Star not found.")

In [ ]:
file_name = "test.hdf5"

with h5py.File(file_name, "w") as file:
            
    grp_wfs = file.create_group("WFS")
    grp_wfs.attrs["Loop_Gain"] = loop_gain
    grp_wfs.attrs["Loop_Leak"] = loop_leak
    grp_wfs.attrs["Loop_Freq"] = wfs_fps

    grp_wfs.create_dataset("WFS_Images", data=wfs_frames)
    grp_wfs.create_dataset("Dark", data=wfs_frames[0]*0)
    grp_wfs.create_dataset("Reference_Frame", data=wfs_frames[0]*0)

    grp_wfs.create_dataset("Valid_Pixel_Map", data=WFS_PUP)
    grp_wfs.create_dataset("DM_commands", data=dm_commands)
    grp_wfs.create_dataset("DM_TimeStamps", data=dm_timestamps)

    grp_wfs.create_dataset("DM_flat", data=dm_commands[0]*0)
    grp_wfs.create_dataset("DM_offset", data=dm_commands[0]*0)
    grp_wfs.create_dataset("WFS_measurements", data=dmResidual)

    grp_science = file.create_group("Science")
    grp_science.attrs["loop_status"] = loop_status_list
    grp_science.attrs["Target"] = target
    grp_science.attrs["Elevation"] = alt
    grp_science.attrs["PSF_TimeStamps"] = psf_timestamps

    grp_science.attrs["Vmag"] = Vmag
    grp_science.attrs["Rmag"] = Rmag
    grp_science.attrs["Jmag"] = Jmag
    grp_science.attrs["Hmag"] = Hmag

    dset_science = grp_science.create_dataset("Science_PSFs", data=psf_frames)
    dset_science.attrs["Exposure_Time"] = sci_dit
    dset_science.attrs["FPS"] = sci_fps
    dset_science.attrs["Gain"] = sci_gain
    dset_science.attrs["Sampling"] = config['science_camera']['sampling_at_calibration']
    dset_science.attrs["Wavelength"] = config['science_camera']['wvl_nm']*1e-9
    dset_science.attrs["Bandpass"] = config['science_camera']['bandpass_nm']*1e-9
    dset_science_dark = grp_science.create_dataset("Dark", data=psf_frames[0]*0)


    grp_calibration = file.create_group("Calibration")
    dset_iMat = grp_calibration.create_dataset("Interaction_Matrix", data=calib_CL.D)
    dset_iMat.attrs["Wavelength"] = config['wfs']['interaction_matrix_wvl_nm']*1e-9
    grp_calibration.create_dataset("M2C", data=M2C_CL)
    grp_calibration.create_dataset("C2Z", data=C2Z)
    grp_calibration.create_dataset("DM_modes", data=dm.modes.transpose(1,0).reshape(-1, tel.resolution, tel.resolution))
    grp_calibration.create_dataset("Z_full_resolution", data=Z.modesFullRes.transpose(2,0,1))
    grp_calibration.attrs["Diameter"] = config['telescope']['diameter_m']
    grp_calibration.attrs["Obstruction_ratio"] = config['telescope']['obstruction_ratio']
    grp_calibration.attrs["Science_Calibration_Wavelength"] = config['science_camera']['calibration_wvl_nm']*1e-9
    grp_calibration.attrs["AO_Calibration_Wavelength"] = config['wfs']['interaction_matrix_wvl_nm']*1e-9
    grp_calibration.attrs["SkyCalibPupilRatio"] = config['dm']['sky_calib_pupil_ratio']
    grp_calibration.attrs["Actuators_in_diameter"] = config['dm']['actuators_in_diameter']
    grp_calibration.attrs["Total_Number_Of_Actuators"] = dm.nValidAct
    grp_calibration.attrs["Total_Number_Of_Controlled_Modes"] = M2C_CL.shape[1]


In [ ]:
dm.modes.transpose(1,0).reshape(-1, tel.resolution, tel.resolution).shape

In [ ]:
plt.imshow(np.log(np.abs(np.mean(psf_frames, axis = 0))))

In [ ]:
(dm_commands @ C2Z.T).shape

In [ ]:
plt.plot(dm_commands[0])

In [ ]:
plt.plot(C2Z[0])

In [ ]:
atm